# Apriori
        **unsupervised › association_rules** · notebook `apriori`

        **Kapan dipakai:** Algoritma klasik itemset sering; mudah dijelaskan; baik untuk data kecil-menengah.

        **Jebakan umum:** Lambat bila support rendah atau item banyak (ledakan kombinasi).

**Data demo:** Groceries (9835 transaksi belanja).

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
PATH = ROOT / "data_latihan/unsupervised/groceries.csv"
MIN_SUPPORT = 0.01     # itemset harus muncul di >= 1% transaksi
MIN_CONF = 0.2           # aturan: P(konsekuen | anteseden) >= 20%
MODEL_KEY = "apriori"
OUT = out_dir("unsupervised", "association_rules", MODEL_KEY)

## 2. Transaksi → matriks biner
Tiap baris = satu transaksi; tiap kolom = satu item (True bila ada).

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import association_rules
from superdb.unsupervised import load_basket
tx = load_basket(PATH); te = TransactionEncoder(); onehot = pd.DataFrame(te.fit(tx).transform(tx), columns=te.columns_)
print(onehot.shape, "| rata-rata item per transaksi:", round(onehot.sum(1).mean(), 2))
display(onehot.mean().sort_values(ascending=False).head(8).round(3).to_frame("support"))

## 3. Itemset sering

In [ ]:
from mlxtend.frequent_patterns import apriori

algo = lambda df, s: apriori(df, min_support=s, use_colnames=True)
import time; t0 = time.time(); freq = algo(onehot, MIN_SUPPORT); print(f"{len(freq)} itemset sering dalam {time.time() - t0:.1f} dtk")
freq["panjang"] = freq["itemsets"].map(len); display(freq.sort_values("support", ascending=False).head(8))

## 4. Aturan asosiasi
- **support**: seberapa sering anteseden+konsekuen muncul bersama.
- **confidence**: P(konsekuen | anteseden).
- **lift** > 1: muncul bersama lebih sering dari kebetulan (lift = 1 berarti independen). Urutkan dengan lift, bukan hanya confidence.

In [ ]:
try:
    rules = association_rules(freq, num_itemsets=len(onehot), metric="confidence", min_threshold=MIN_CONF)
except TypeError:
    rules = association_rules(freq, metric="confidence", min_threshold=MIN_CONF)
rules = rules[rules["lift"] > 1.0].sort_values("lift", ascending=False)
show = rules.assign(anteseden=rules.antecedents.map(lambda s: ", ".join(sorted(s))), konsekuen=rules.consequents.map(lambda s: ", ".join(sorted(s))))
display(show[["anteseden", "konsekuen", "support", "confidence", "lift"]].head(10).round(3))
metrics = {"n_itemset": int(len(freq)), "n_aturan_lift>1": int(len(rules)), "lift_maks": float(rules["lift"].max()) if len(rules) else None}
fig, ax = plt.subplots(figsize=(5.5, 4)); sc = ax.scatter(rules["support"], rules["confidence"], c=rules["lift"], s=14, cmap="viridis"); plt.colorbar(sc, label="lift")
ax.set(xlabel="support", ylabel="confidence", title="Aturan asosiasi"); plt.tight_layout(); plt.savefig(OUT / "rules.png", dpi=130); plt.show()

In [ ]:
(rules.drop(columns=["antecedents", "consequents"], errors="ignore").assign(antecedents=rules.antecedents.map(lambda s: "|".join(sorted(s))), consequents=rules.consequents.map(lambda s: "|".join(sorted(s)))) if len(rules) else rules).to_csv(OUT / "rules.csv", index=False)
save_json({"metrics": metrics, "min_support": MIN_SUPPORT, "min_conf": MIN_CONF}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Turunkan `MIN_SUPPORT` untuk menemukan aturan langka (jumlah itemset melonjak).
- Aturan dengan lift tinggi tetapi support sangat kecil bisa kebetulan; periksa jumlah transaksi pendukung.
- Aturan menunjukkan asosiasi, **bukan sebab-akibat**.